# Low-read-excluded primary continuous analyses

These fits exclude patient × cell-type profiles with fewer than 1,000 allocated reads, retaining the original score and covariate scaling and patient basis. The all-profile models are the sensitivity comparisons.

Set BASE, PCA_DIR, SENS_ROOT and the original_run entries in the embedded engine to the archived inputs and separate output directories. The final cell selects FAMILIES; its preserved setting is immune_fine. Choose immune_coarse and nonimmune as needed. The engine is written to /content/lowread_engine.py; ENGINE_PATH must match that location. Execute the four code cells in order.


In [ ]:
# 1. Install the validated sampling environment
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pymc==5.28.5", "arviz==0.23.4", "nutpie==0.16.11"])
print("Pinned environment installed.")


In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%%writefile /content/lowread_engine.py
# 3. Write the analysis engine (same code as BHM_lowread_sensitivity_v1)
# Engine section: configuration (values come from CONFIG set by the driver)
FAMILY = CONFIG["family"]
PROFILE = CONFIG["profile"]
STAGE = CONFIG["stage"]
MIN_READS = CONFIG.get("min_reads", 1000)
LOCAL_PATHS = CONFIG.get("local_paths")
SAMPLER_OVERRIDE = CONFIG.get("sampler_override")
PPC_DRAWS_OVERRIDE = CONFIG.get("ppc_draws_override")
PROMOTION_WAIVER = CONFIG.get("waiver")

import gc, hashlib, json, math, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "lowread-sensitivity-v1"
SCORE_VARIANT = "within_component_logcpm_z_cap5"
BASE = Path("/content/drive/MyDrive/new_dysfunction")
PCA_DIR = BASE / ("bayesprism_initial_issue_verification_v1/publication_pca_tiered_qc_v1/"
                  "20260906T013040524557Z")
SENS_ROOT = BASE / "normalized_state_bhm_lowread_sensitivity_v1"

FAMILY_SPECS = {
    "immune_coarse": {
        "compartment": "immune_course", "tier_column": "tiered_primary_keep",
        "nu_scope": "celltype", "architecture": "v2p2_g1_fixed_ctnu",
        "role": "primary", "expected_features": 306, "expected_celltypes": 8,
        "expected_dropped_cells": 25,
        "original_run": BASE / "normalized_state_bhm_v2p2_celltype_nu" / (
            "v2p2_g1_fixed_ctnu__immune_coarse_primary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "immune_coarse_v2p2_feature_bmi_posterior.csv",
    },
    "nonimmune": {
        "compartment": "non_immune", "tier_column": "strict_qc_keep",
        "nu_scope": "global", "architecture": "v2p1c_g1_fixed_gnu",
        "role": "primary", "expected_features": 112, "expected_celltypes": 4,
        "expected_dropped_cells": 2,
        "original_run": BASE / "normalized_state_bhm_v2p1c_separate_compartments" / (
            "v2p1c_g1_fixed_gnu__nonimmune_primary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "nonimmune_v2p1c_feature_bmi_posterior.csv",
    },
    "immune_fine": {
        "compartment": "immune_fine", "tier_column": "tiered_primary_keep",
        "nu_scope": "global", "architecture": "v2p1c_g1_fixed_gnu",
        "role": "secondary", "expected_features": 496, "expected_celltypes": 14,
        "expected_dropped_cells": 58,
        "original_run": BASE / "normalized_state_bhm_v2p1c_separate_compartments" / (
            "v2p1c_g1_fixed_gnu__immune_fine_secondary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "immune_fine_v2p1c_feature_bmi_posterior.csv",
    },
}

PROFILES = {
    "smoke": {"tune": 500, "draws": 500, "chains": 2, "target_accept": 0.97},
    "standard": {"tune": 1000, "draws": 1000, "chains": 4, "target_accept": 0.98},
    "publication": {"tune": 2000, "draws": 4000, "chains": 4, "target_accept": 0.99},
}
PREDECESSOR = {"smoke": None, "standard": "smoke", "publication": "standard"}
PPC_DRAWS = {"smoke": 200, "standard": 2000, "publication": 4000}

BASELINE_SD = 0.50
CELLTYPE_BMI_SD = 0.20
FEATURE_BMI_SD = 0.30
CELLTYPE_COVARIATE_SD = 0.25
FEATURE_COVARIATE_SD = 0.25
PATIENT_GLOBAL_SD = 0.50
LOG_SIGMA_LOCATION = math.log(0.65)
LOG_SIGMA_LOCATION_SD = 0.40
CELLTYPE_LOG_SIGMA_SD = 0.25
FEATURE_LOG_SIGMA_SD = 0.50
NU_MINUS_TWO_RATE = 0.10

PRIMARY_ROPE = 0.10
ROPE_GRID = (0.05, 0.10, 0.15, 0.20)
Q_LEVEL = 0.05
MAX_TREE_DEPTH = 10
MAX_TREE_DEPTH_FRACTION = 0.01
BFMI_MIN = 0.30

# (SMOKE_GLOBAL_RHAT_HARD_MAX = 1.10). The first sequential run used 1.05, which stopped
# immune fine at R-hat 1.0506 on a log-scale hyperparameter with clean geometry.
SMOKE_RHAT_HARD_MAX = 1.10
GATES = {
    "smoke": {"rhat_max": 1.05, "ess_min": 100},
    "standard": {"rhat_max": 1.01, "ess_min": 400},
    "publication": {"rhat_max": 1.01, "ess_min": 400},
}
PPC_ALPHA = 0.025
PPC_CORRELATION_THRESHOLD = 0.90
PPC_BATCH = 32

# Hashes recorded by the final fits' analysis locks.
LOCKED_HASHES = {
    "pca_manifest.json": "a4263b46efb1c0d1d98a01b127ddb143bb7616ce88f8b364959613bbb74dcd30",
    "component_inclusion_tiers.csv": "16494a18ed467223134f0aed2efbf63c084d2f6b09332ca9854a9c403dd67294",
    "pca_input_hashes.csv": "6e2eaedbb5efe7c00d82e82c939d4416a5a3853f14bde881a55471cf874259c9",
    "combined_review_manifest.json": "04b0b1d788fc69e78ded4b44e050d487bfe231dfd5e5bd2b34f6b359c74f4476",
}
DECISION_RULE = {
    "any_bfdr_selection_change": True,
    "sign_flip_among_hdi_excluding_zero_features": True,
    "hdi_status_change_fraction_max": 0.10,
    "hdi_status_change_count_min": 3,
    "celltype_mean_slope_shift_max": 0.05,
}

if FAMILY not in FAMILY_SPECS:
    raise ValueError(f"FAMILY must be one of {sorted(FAMILY_SPECS)}")
if PROFILE not in PROFILES:
    raise ValueError(f"PROFILE must be one of {sorted(PROFILES)}")
if STAGE not in {"preflight", "fit", "diagnostics", "ppc", "compare"}:
    raise ValueError("Unknown STAGE")
if STAGE == "compare" and PROFILE != "publication":
    raise ValueError("compare uses the publication fit; set PROFILE = 'publication'")
if MIN_READS != 1000:
    warnings.warn("MIN_READS differs from the prespecified 1,000-read threshold.")

SPEC = FAMILY_SPECS[FAMILY]
RUN_NAME = f"{SPEC['architecture']}__{FAMILY}__lowread{MIN_READS}"
RUN_SEED = 20260916 + int(hashlib.sha256(RUN_NAME.encode()).hexdigest()[:8], 16) % 1_000_000
ORIGINAL_RUN = Path(SPEC["original_run"])
if LOCAL_PATHS:
    PCA_DIR = Path(LOCAL_PATHS["pca_dir"])
    SENS_ROOT = Path(LOCAL_PATHS["sens_root"])
    ORIGINAL_RUN = Path(LOCAL_PATHS["original_runs"][FAMILY])
RUN_ROOT = SENS_ROOT / RUN_NAME
OUTPUT_DIR = RUN_ROOT / PROFILE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRACE_PATH = OUTPUT_DIR / "posterior_trace.nc"
print(f"{RUN_NAME} | profile {PROFILE} | stage {STAGE} | seed {RUN_SEED}")
print("Output:", OUTPUT_DIR)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(value, path):
    tmp = Path(path).with_suffix(".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def write_csv(frame, path):
    tmp = Path(str(path) + ".tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def _clean_attrs(attrs):
    for key in list(attrs):
        value = attrs[key]
        if value is None or isinstance(value, (dict, set, bool)) or (
                isinstance(value, (list, tuple)) and any(isinstance(v, (dict, list, tuple, type(None))) for v in value)):
            attrs[key] = json.dumps(value, default=str, sort_keys=True)

def sanitize_attrs(idata):
    """netCDF rejects nested metadata (nutpie stores a dict); store it as JSON text."""
    _clean_attrs(idata.attrs)
    for group in idata.groups():
        _clean_attrs(idata[group].attrs)

def strict_bool(series):
    mapped = series.map({True: True, False: False, "True": True, "False": False,
                         "true": True, "false": False})
    if mapped.isna().any():
        raise ValueError("Non-boolean inclusion flag")
    return mapped.astype(bool)

# 4. Load the frozen inputs and apply the low-read rule
for name, expected in LOCKED_HASHES.items():
    folder = PCA_DIR
    if name == "combined_review_manifest.json":
        continue  # checked after resolving the review directory
    actual = sha256_file(folder / name)
    if actual != expected:
        raise RuntimeError(f"{name} does not match the locked hash")

pca_manifest = json.loads((PCA_DIR / "pca_manifest.json").read_text())
REVIEW_DIR = Path(pca_manifest["combined_review_directory"])
if LOCAL_PATHS:
    REVIEW_DIR = Path(LOCAL_PATHS["review_dir"])
review_manifest_path = REVIEW_DIR / "combined_review_manifest.json"
if sha256_file(review_manifest_path) != LOCKED_HASHES["combined_review_manifest.json"]:
    raise RuntimeError("combined_review_manifest.json does not match the locked hash")
review_manifest = json.loads(review_manifest_path.read_text())
RUN_DIR = Path(review_manifest["selected_runs"][SPEC["compartment"]])
if LOCAL_PATHS:
    RUN_DIR = Path(LOCAL_PATHS["run_dirs"][SPEC["compartment"]])

input_hashes = pd.read_csv(PCA_DIR / "pca_input_hashes.csv")
expected_input = input_hashes[input_hashes["compartment"].eq(SPEC["compartment"])]
cohort_path = RUN_DIR / "analysis_cohort.csv"
score_path = RUN_DIR / "audit_scores_all_variants.csv.gz"
qc_path = RUN_DIR / "patient_qc.csv"
for kind, path in (("cohort", cohort_path), ("scores", score_path)):
    locked = expected_input.loc[expected_input["input"].eq(kind), "sha256"]
    if len(locked) != 1 or sha256_file(path) != locked.iloc[0]:
        raise RuntimeError(f"{path.name} does not match pca_input_hashes.csv")

cohort = pd.read_csv(cohort_path, dtype={"base_sample_id": str})
scores = pd.read_csv(score_path, dtype={"base_sample_id": str})
patient_qc = pd.read_csv(qc_path, dtype={"base_sample_id": str})
tiers = pd.read_csv(PCA_DIR / "component_inclusion_tiers.csv")

if cohort["base_sample_id"].nunique() != 127:
    raise ValueError("Expected 127 patients")
if cohort["bmi_group"].value_counts().to_dict() != {"overweight": 57, "normal_weight": 52, "obese": 18}:
    raise ValueError("BMI group counts changed")

local_tiers = tiers[tiers["compartment"].eq(SPEC["compartment"])]
keep_celltypes = sorted(local_tiers.loc[strict_bool(local_tiers[SPEC["tier_column"]]), "CellType"].astype(str))
covariate_columns = ["base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"]
frame = scores[scores["variant"].eq(SCORE_VARIANT) & scores["CellType"].isin(keep_celltypes)].copy()
frame["global_celltype"] = SPEC["compartment"] + "||" + frame["CellType"].astype(str)
frame["global_feature"] = SPEC["compartment"] + "||" + frame["feature"].astype(str)
frame = frame.merge(cohort[covariate_columns], on="base_sample_id", how="inner", validate="many_to_one")
if frame.duplicated(["base_sample_id", "global_feature"]).any():
    raise ValueError("Duplicate patient/feature rows")

# Standardise on all 127 patients, exactly as the final fits did.
frame["score"] = pd.to_numeric(frame["score"], errors="coerce")
grouped = frame.groupby("global_feature")["score"]
frame["score_model"] = (frame["score"] - grouped.transform("mean")) / grouped.transform("std")
frame["bmi_standardized"] = frame["bmi_z"]
if not np.isfinite(frame[["score_model", "bmi_standardized", "age_z", "sex_centered"]].to_numpy(float)).all():
    raise ValueError("Nonfinite model values")
if not frame.groupby("global_feature")["base_sample_id"].nunique().eq(127).all():
    raise ValueError("Every feature must contain all 127 patients before the low-read rule")

n_features = frame["global_feature"].nunique()
n_celltypes = frame["global_celltype"].nunique()
if (n_features, n_celltypes) != (SPEC["expected_features"], SPEC["expected_celltypes"]):
    raise ValueError(f"Expected {SPEC['expected_features']} features / {SPEC['expected_celltypes']} "
                     f"cell types, found {n_features} / {n_celltypes}")

original_inventory_path = ORIGINAL_RUN / "feature_inventory.csv"
if original_inventory_path.exists():
    original_features = set(pd.read_csv(original_inventory_path)["global_feature"])
    if original_features != set(frame["global_feature"]):
        raise ValueError("Feature set differs from the final fit's feature_inventory.csv")
    print("Feature set matches the final fit.")
else:
    warnings.warn(f"Cannot check against {original_inventory_path}")

# Low-read rule.
reads = patient_qc[["base_sample_id", "CellType", "component_total", "theta"]].copy()
reads["component_total"] = pd.to_numeric(reads["component_total"], errors="coerce")
frame = frame.merge(reads, on=["base_sample_id", "CellType"], how="left", validate="many_to_one")
if frame["component_total"].isna().any():
    raise ValueError("Missing allocated-read totals")
frame["low_read"] = frame["component_total"] < MIN_READS

dropped_cells = (frame.loc[frame["low_read"], ["base_sample_id", "CellType", "component_total", "theta", "bmi"]]
                 .drop_duplicates(["base_sample_id", "CellType"])
                 .sort_values(["CellType", "component_total"]).reset_index(drop=True))
n_dropped_obs = int(frame["low_read"].sum())
if len(dropped_cells) != SPEC["expected_dropped_cells"] and MIN_READS == 1000:
    raise ValueError(f"Expected {SPEC['expected_dropped_cells']} low-read cells, found {len(dropped_cells)}")

full_frame = frame.copy()                       # complete grid, used for the observed-data PPC mask
analysis_data = frame.loc[~frame["low_read"]].reset_index(drop=True)
per_feature = analysis_data.groupby("global_feature")["base_sample_id"].nunique()
if per_feature.min() < 30:
    raise ValueError(f"A feature keeps only {per_feature.min()} patients")
if analysis_data["base_sample_id"].nunique() != 127:
    raise ValueError("A patient lost every observation")

write_csv(dropped_cells, RUN_ROOT / "dropped_low_read_cells.csv")
preflight = {
    "notebook_version": NOTEBOOK_VERSION, "family": FAMILY, "role": SPEC["role"],
    "architecture": SPEC["architecture"], "nu_scope": SPEC["nu_scope"],
    "min_reads": MIN_READS, "seed": RUN_SEED,
    "n_patients": 127, "n_features": n_features, "n_celltypes": n_celltypes,
    "n_observations_full": len(full_frame), "n_observations_fitted": len(analysis_data),
    "n_dropped_cells": len(dropped_cells), "n_dropped_observations": n_dropped_obs,
    "dropped_cells_by_celltype": dropped_cells["CellType"].value_counts().to_dict(),
    "minimum_patients_per_feature": int(per_feature.min()),
    "standardisation": "127-patient feature mean/SD from the final fits (before removal)",
    "input_sha256": {"cohort": sha256_file(cohort_path), "scores": sha256_file(score_path),
                     "patient_qc": sha256_file(qc_path)},
    "decision_rule": DECISION_RULE,
}
print(json.dumps(preflight, indent=2, default=str))
display_cells = dropped_cells.copy()
display_cells["component_total"] = display_cells["component_total"].round(1)
print(display_cells.to_string(index=False))

# 5. Build the model (final architecture, incomplete grid)
import pymc as pm
import pytensor.tensor as pt
import arviz as az

def prepare_arrays(data):
    data = data.sort_values(["global_celltype", "global_feature", "base_sample_id"]).reset_index(drop=True).copy()
    features = sorted(data["global_feature"].unique())
    celltypes = sorted(data["global_celltype"].unique())
    patients = sorted(full_frame["base_sample_id"].unique())
    f_index = {n: i for i, n in enumerate(features)}
    c_index = {n: i for i, n in enumerate(celltypes)}
    p_index = {n: i for i, n in enumerate(patients)}
    data["feature_idx"] = data["global_feature"].map(f_index)
    data["celltype_idx"] = data["global_celltype"].map(c_index)
    data["patient_idx"] = data["base_sample_id"].map(p_index)
    mapping = data[["feature_idx", "celltype_idx"]].drop_duplicates().sort_values("feature_idx")
    if len(mapping) != len(features):
        raise ValueError("Every feature must map to one cell type")
    feature_to_celltype = mapping["celltype_idx"].to_numpy(int)

    # Patient design and orthogonal residual basis from all 127 patients (unchanged).
    predictors = ["bmi_standardized", "age_z", "sex_centered"]
    table = (full_frame[["base_sample_id", *predictors]].drop_duplicates("base_sample_id")
             .set_index("base_sample_id").loc[patients])
    design = np.column_stack([np.ones(len(patients)), table.to_numpy(float)])
    if np.linalg.matrix_rank(design) != design.shape[1]:
        raise ValueError("Patient design is rank deficient")
    u, s, _ = np.linalg.svd(design, full_matrices=True)
    rank = int((s > max(design.shape) * np.finfo(float).eps * s.max()).sum())
    basis = u[:, rank:] * np.sqrt(len(patients) / (len(patients) - rank))
    if np.abs(design.T @ basis).max() > 1e-10:
        raise AssertionError("Patient basis is not orthogonal")
    return {"data": data, "features": features, "celltypes": celltypes, "patients": patients,
            "feature_to_celltype": feature_to_celltype, "patient_basis": basis,
            "design_rank": rank}

def build_model(arrays):
    data = arrays["data"]
    covariates = ["age_z", "sex_centered"]
    coords = {"obs_id": np.arange(len(data)), "feature": arrays["features"],
              "celltype": arrays["celltypes"], "patient": arrays["patients"],
              "patient_re_dimension": np.arange(arrays["patient_basis"].shape[1]),
              "covariate": covariates}
    with pm.Model(coords=coords) as model:
        feature_idx = pm.Data("feature_idx", data["feature_idx"].to_numpy(int), dims="obs_id")
        patient_idx = pm.Data("patient_idx", data["patient_idx"].to_numpy(int), dims="obs_id")
        celltype_idx = pm.Data("celltype_idx", data["celltype_idx"].to_numpy(int), dims="obs_id")
        f2c = pm.Data("feature_to_celltype", arrays["feature_to_celltype"], dims="feature")
        bmi_z = pm.Data("bmi_z", data["bmi_standardized"].to_numpy(float), dims="obs_id")
        cov = pm.Data("covariate_matrix", data[covariates].to_numpy(float), dims=("obs_id", "covariate"))
        y = pm.Data("y", data["score_model"].to_numpy(float), dims="obs_id")
        basis = pm.Data("patient_residual_basis", arrays["patient_basis"],
                        dims=("patient", "patient_re_dimension"))

        feature_baseline = pm.Normal("feature_baseline", 0.0, BASELINE_SD, dims="feature")
        celltype_bmi_slope = pm.Normal("celltype_bmi_slope", 0.0, CELLTYPE_BMI_SD, dims="celltype")
        feature_sigma_bmi = pm.HalfNormal("feature_sigma_bmi", FEATURE_BMI_SD, dims="celltype")
        feature_bmi_raw = pm.Normal("feature_bmi_raw", 0.0, 1.0, dims="feature")
        feature_bmi_slope = pm.Deterministic(
            "feature_bmi_slope",
            celltype_bmi_slope[f2c] + feature_bmi_raw * feature_sigma_bmi[f2c], dims="feature")

        celltype_covariate_slope = pm.Normal("celltype_covariate_slope", 0.0, CELLTYPE_COVARIATE_SD,
                                             dims=("covariate", "celltype"))
        feature_sigma_covariate = pm.HalfNormal("feature_sigma_covariate", FEATURE_COVARIATE_SD,
                                                dims=("covariate", "celltype"))
        feature_covariate_raw = pm.Normal("feature_covariate_raw", 0.0, 1.0, dims=("covariate", "feature"))
        feature_covariate_slope = pm.Deterministic(
            "feature_covariate_slope",
            celltype_covariate_slope[:, f2c] + feature_covariate_raw * feature_sigma_covariate[:, f2c],
            dims=("covariate", "feature"))

        patient_global_sigma = pm.HalfNormal("patient_global_sigma", PATIENT_GLOBAL_SD)
        patient_global_coefficient = pm.Normal("patient_global_coefficient", 0.0, patient_global_sigma,
                                               dims="patient_re_dimension")
        patient_global_effect = pm.Deterministic(
            "patient_global_effect", pt.dot(basis, patient_global_coefficient), dims="patient")

        mu = (feature_baseline[feature_idx] + feature_bmi_slope[feature_idx] * bmi_z
              + patient_global_effect[patient_idx]
              + pt.sum(cov * feature_covariate_slope[:, feature_idx].T, axis=1))

        log_sigma_location = pm.Normal("log_sigma_location", LOG_SIGMA_LOCATION, LOG_SIGMA_LOCATION_SD)
        log_sigma_celltype_location_sd = pm.HalfNormal("log_sigma_celltype_location_sd", CELLTYPE_LOG_SIGMA_SD)
        log_sigma_celltype_raw = pm.Normal("log_sigma_celltype_raw", 0.0, 1.0, dims="celltype")
        log_sigma_celltype_location = pm.Deterministic(
            "log_sigma_celltype_location",
            log_sigma_location + log_sigma_celltype_location_sd * log_sigma_celltype_raw, dims="celltype")
        log_sigma_feature_scale = pm.HalfNormal("log_sigma_feature_scale", FEATURE_LOG_SIGMA_SD, dims="celltype")
        log_sigma_raw = pm.Normal("log_sigma_raw", 0.0, 1.0, dims="feature")
        sigma_feature = pm.Deterministic(
            "sigma_feature",
            pt.exp(log_sigma_celltype_location[f2c] + log_sigma_feature_scale[f2c] * log_sigma_raw),
            dims="feature")

        if SPEC["nu_scope"] == "celltype":
            nu_minus_two = pm.Exponential("nu_minus_two", lam=NU_MINUS_TWO_RATE, dims="celltype")
            nu = pm.Deterministic("nu", nu_minus_two + 2.0, dims="celltype")
            nu_obs = nu[celltype_idx]
        else:
            nu_minus_two = pm.Exponential("nu_minus_two", lam=NU_MINUS_TWO_RATE)
            nu = pm.Deterministic("nu", nu_minus_two + 2.0)
            nu_obs = nu
        pm.StudentT("y_obs", nu=nu_obs, mu=mu, sigma=sigma_feature[feature_idx], observed=y, dims="obs_id")
    return model

arrays = prepare_arrays(analysis_data)
model = build_model(arrays)
initial_logp = float(model.compile_logp()(model.initial_point()))
if not np.isfinite(initial_logp):
    raise ValueError("Initial log density is not finite")
expected_rvs = {"feature_baseline", "celltype_bmi_slope", "feature_sigma_bmi", "feature_bmi_raw",
                "celltype_covariate_slope", "feature_sigma_covariate", "feature_covariate_raw",
                "patient_global_sigma", "patient_global_coefficient", "log_sigma_location",
                "log_sigma_celltype_location_sd", "log_sigma_celltype_raw",
                "log_sigma_feature_scale", "log_sigma_raw", "nu_minus_two"}
if {rv.name for rv in model.free_RVs} != expected_rvs:
    raise AssertionError("Free random variables differ from the final architecture")
if (model["nu"].ndim == 1) != (SPEC["nu_scope"] == "celltype"):
    raise AssertionError("nu scope does not match the final architecture")

structure = {**preflight, "initial_logp": initial_logp,
             "patient_basis_dimensions": int(arrays["patient_basis"].shape[1]),
             "free_rvs": sorted(expected_rvs)}
write_json(structure, RUN_ROOT / "preflight_manifest.json")
print(f"Model built: {len(arrays['data']):,} observations, initial logp {initial_logp:.1f}")

# 6. Fit (with enforced promotion) and convergence gates
def gate_path(profile):
    return RUN_ROOT / profile / "run_diagnostics.json"

def compute_diagnostics(idata, profile):
    post = idata.posterior
    names = [v for v in post.data_vars]
    rhat = az.rhat(post[names])
    ess_bulk = az.ess(post[names], method="bulk")
    ess_tail = az.ess(post[names], method="tail")
    rows = []
    for name in names:
        r = np.atleast_1d(rhat[name].values).ravel()
        b = np.atleast_1d(ess_bulk[name].values).ravel()
        t = np.atleast_1d(ess_tail[name].values).ravel()
        rows.append({"parameter_family": name, "n": r.size, "max_rhat": float(np.nanmax(r)),
                     "min_ess_bulk": float(np.nanmin(b)), "min_ess_tail": float(np.nanmin(t)),
                     "nonfinite": int((~np.isfinite(r)).sum() + (~np.isfinite(b)).sum())})
    families = pd.DataFrame(rows).sort_values("max_rhat", ascending=False)
    stats = idata.sample_stats
    divergences = int(stats["diverging"].values.sum())
    bfmi = [float(x) for x in np.atleast_1d(az.bfmi(idata))]
    if "maxdepth_reached" in stats:
        maxdepth_fraction = stats["maxdepth_reached"].values.reshape(len(bfmi), -1).mean(axis=1)
    else:
        maxdepth_fraction = (stats["tree_depth"].values >= MAX_TREE_DEPTH).reshape(len(bfmi), -1).mean(axis=1)
    gate = GATES[profile]
    record = {
        "family": FAMILY, "profile": profile, "run_name": RUN_NAME,
        "chains": int(post.sizes["chain"]), "draws_per_chain": int(post.sizes["draw"]),
        "divergences": divergences, "min_bfmi": min(bfmi),
        "max_maxdepth_fraction": float(np.max(maxdepth_fraction)),
        "max_rhat": float(families["max_rhat"].max()),
        "min_ess_bulk": float(families["min_ess_bulk"].min()),
        "min_ess_tail": float(families["min_ess_tail"].min()),
        "nonfinite_diagnostics": int(families["nonfinite"].sum()),
        "trace_sha256": sha256_file(TRACE_PATH) if TRACE_PATH.exists() else None,
        "code_test_override": SAMPLER_OVERRIDE,
        "promotion_waiver": (json.loads((OUTPUT_DIR / "promotion_waiver.json").read_text())["reason"]
                             if (OUTPUT_DIR / "promotion_waiver.json").exists() else None),
    }
    record["gate_pass"] = bool(
        SAMPLER_OVERRIDE is None and divergences == 0 and min(bfmi) >= BFMI_MIN
        and record["max_maxdepth_fraction"] <= MAX_TREE_DEPTH_FRACTION
        and record["nonfinite_diagnostics"] == 0
        and record["max_rhat"] < gate["rhat_max"]
        and record["min_ess_bulk"] >= gate["ess_min"] and record["min_ess_tail"] >= gate["ess_min"])
    return record, families

idata = None
if STAGE == "fit":
    predecessor = PREDECESSOR[PROFILE]
    if predecessor is not None and SAMPLER_OVERRIDE is None:
        path = gate_path(predecessor)
        passed = path.exists() and json.loads(path.read_text()).get("gate_pass")
        if not passed:
            if not PROMOTION_WAIVER:
                raise RuntimeError(f"{PROFILE} requires a passed {predecessor} gate ({path}); "
                                   "see PROMOTION_WAIVER in cell 3")
            smoke_file = gate_path("smoke")
            if not smoke_file.exists():
                raise RuntimeError("A waiver needs a completed smoke fit")
            smoke = json.loads(smoke_file.read_text())
            geometry_ok = (smoke["divergences"] == 0 and smoke["min_bfmi"] >= BFMI_MIN
                           and smoke["max_maxdepth_fraction"] <= MAX_TREE_DEPTH_FRACTION
                           and smoke["nonfinite_diagnostics"] == 0 and smoke["max_rhat"] < SMOKE_RHAT_HARD_MAX
                           and smoke["code_test_override"] is None)
            if not geometry_ok:
                raise RuntimeError("Waiver refused: the smoke fit has a sampler-geometry problem")
            write_json({"profile": PROFILE, "failed_or_missing_predecessor": predecessor,
                        "predecessor_record": json.loads(path.read_text()) if path.exists() else None,
                        "smoke_record": smoke, "reason": PROMOTION_WAIVER,
                        "binding_gate": "publication: R-hat < 1.01, bulk/tail ESS >= 400, 0 divergences"},
                       OUTPUT_DIR / "promotion_waiver.json")
            print("Promotion waiver recorded:", PROMOTION_WAIVER)
    if TRACE_PATH.exists():
        raise FileExistsError(f"{TRACE_PATH} exists; never overwrite a trace")
    sampler = dict(PROFILES[PROFILE])
    if SAMPLER_OVERRIDE:
        sampler.update(SAMPLER_OVERRIDE)
        warnings.warn("SAMPLER_OVERRIDE is set: this run is a code test and can never pass a gate")
    start = time.time()
    with model:
        idata = pm.sample(tune=sampler["tune"], draws=sampler["draws"], chains=sampler["chains"],
                          cores=sampler["chains"], target_accept=sampler["target_accept"],
                          nuts_sampler="nutpie", nuts_sampler_kwargs={"maxdepth": MAX_TREE_DEPTH},
                          random_seed=RUN_SEED, return_inferencedata=True, progressbar=False)
    elapsed = time.time() - start
    idata.attrs.update({"run_name": RUN_NAME, "profile": PROFILE, "notebook_version": NOTEBOOK_VERSION})
    sanitize_attrs(idata)
    tmp = TRACE_PATH.with_name(".tmp_" + TRACE_PATH.name)
    idata.to_netcdf(tmp)
    tmp.replace(TRACE_PATH)
    print(f"Sampling took {elapsed / 60:.1f} min")
elif STAGE in {"diagnostics", "ppc", "compare"}:
    trace_profile = "publication" if STAGE == "compare" else PROFILE
    path = RUN_ROOT / trace_profile / "posterior_trace.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    idata = az.from_netcdf(path)

if STAGE in {"fit", "diagnostics"}:
    diagnostics, family_table = compute_diagnostics(idata, PROFILE)
    write_json(diagnostics, OUTPUT_DIR / "run_diagnostics.json")
    write_csv(family_table, OUTPUT_DIR / "diagnostic_families.csv")
    print(json.dumps(diagnostics, indent=2))
    print(family_table.head(10).to_string(index=False))

# 7. Draw-matched posterior predictive check (incomplete grid)
def nan_residual_stats(matrix):
    """matrix: draws x features x patients with NaN for removed cells."""
    sd = np.nanstd(matrix, axis=2, ddof=1)
    centred = matrix - np.nanmean(matrix, axis=2, keepdims=True)
    tail = np.nanmax(np.abs(centred), axis=2) / sd
    return sd, tail

def dependence_count(matrix, columns):
    x = matrix[:, columns, :]
    if x.shape[1] < 2:
        return np.zeros(x.shape[0], dtype=np.int32)
    x = x - np.nanmean(x, axis=2, keepdims=True)
    x = np.nan_to_num(x, nan=0.0)          # centred on observed cells, zero elsewhere
    norms = np.sqrt((x * x).sum(axis=2))
    corr = np.einsum("bip,bjp->bij", x, x, optimize=True) / (norms[:, :, None] * norms[:, None, :])
    upper = np.triu_indices(x.shape[1], k=1)
    return (np.abs(corr[:, upper[0], upper[1]]) >= PPC_CORRELATION_THRESHOLD).sum(axis=1).astype(np.int32)

def upper_p(reference, observed):
    return float((1 + np.sum(reference >= observed)) / (reference.size + 1))

def lower_p(reference, observed):
    return float((1 + np.sum(reference <= observed)) / (reference.size + 1))

if STAGE == "ppc":
    diag_file = OUTPUT_DIR / "run_diagnostics.json"
    if not diag_file.exists():
        raise RuntimeError("Run the diagnostics for this profile first")
    diag_record = json.loads(diag_file.read_text())
    if not diag_record["gate_pass"] and SAMPLER_OVERRIDE is None:
        raise RuntimeError("PPC requires a trace that passed its convergence gate")

    data = arrays["data"]
    features, patients = arrays["features"], arrays["patients"]
    n_f, n_p, n_c = len(features), len(patients), len(arrays["celltypes"])
    f_idx = data["feature_idx"].to_numpy(int)
    p_idx = data["patient_idx"].to_numpy(int)
    c_idx = data["celltype_idx"].to_numpy(int)
    y_obs = data["score_model"].to_numpy(float)
    bmi = data["bmi_standardized"].to_numpy(float)
    covariates = data[["age_z", "sex_centered"]].to_numpy(float)
    grid = np.full((n_f, n_p), -1, dtype=np.int64)
    grid[f_idx, p_idx] = np.arange(len(data))
    observed_mask = grid >= 0
    columns_by_celltype = [np.flatnonzero(arrays["feature_to_celltype"] == j) for j in range(n_c)]

    def to_grid(values):
        out = np.full((values.shape[0], n_f, n_p), np.nan)
        out[:, observed_mask] = values[:, grid[observed_mask]]
        return out

    post = idata.posterior
    n_chains, n_draws = post.sizes["chain"], post.sizes["draw"]
    requested = PPC_DRAWS_OVERRIDE or PPC_DRAWS[PROFILE]
    requested = min(requested - requested % (2 * n_chains), 2 * n_chains * (n_draws // 2))
    per_chain = requested // n_chains
    positions = np.linspace(0, n_draws - 1, per_chain).round().astype(int)
    chain_of = np.repeat(np.arange(n_chains), per_chain)
    draw_of = np.tile(positions, n_chains)
    rng = np.random.default_rng(RUN_SEED + 70_000)

    raw_obs_grid = to_grid(y_obs[None, :])
    observed_raw_dep = np.array([dependence_count(raw_obs_grid, c)[0] for c in columns_by_celltype])

    stats = {k: [] for k in ("rep_sd", "rep_tail", "obs_sd", "obs_tail")}
    dep = {k: [] for k in ("rep_raw", "rep_res", "obs_res")}
    for start in range(0, requested, PPC_BATCH):
        sel = slice(start, min(start + PPC_BATCH, requested))
        import xarray as xr
        pick = post.isel(chain=xr.DataArray(chain_of[sel], dims="k"), draw=xr.DataArray(draw_of[sel], dims="k"))
        g = lambda name, dims: np.asarray(pick[name].transpose("k", *dims), float)
        base = g("feature_baseline", ("feature",))
        slope = g("feature_bmi_slope", ("feature",))
        cslope = g("feature_covariate_slope", ("covariate", "feature"))
        peff = g("patient_global_effect", ("patient",))
        sigma = g("sigma_feature", ("feature",))
        nu = g("nu", ("celltype",)) if SPEC["nu_scope"] == "celltype" else g("nu", ())[:, None]
        mu = (base[:, f_idx] + slope[:, f_idx] * bmi + peff[:, p_idx]
              + np.einsum("oc,bco->bo", covariates, cslope[:, :, f_idx], optimize=True))
        nu_o = nu[:, c_idx] if SPEC["nu_scope"] == "celltype" else np.broadcast_to(nu, mu.shape)
        y_rep = mu + sigma[:, f_idx] * rng.standard_t(nu_o)

        rep_raw_grid = to_grid(y_rep)
        rep_res_grid = to_grid(y_rep - mu)
        obs_res_grid = to_grid(y_obs[None, :] - mu)
        s1, t1 = nan_residual_stats(rep_res_grid)
        s2, t2 = nan_residual_stats(obs_res_grid)
        stats["rep_sd"].append(s1); stats["rep_tail"].append(t1)
        stats["obs_sd"].append(s2); stats["obs_tail"].append(t2)
        dep["rep_raw"].append(np.column_stack([dependence_count(rep_raw_grid, c) for c in columns_by_celltype]))
        dep["rep_res"].append(np.column_stack([dependence_count(rep_res_grid, c) for c in columns_by_celltype]))
        dep["obs_res"].append(np.column_stack([dependence_count(obs_res_grid, c) for c in columns_by_celltype]))
        del rep_raw_grid, rep_res_grid, obs_res_grid, y_rep, mu
        gc.collect()
    stats = {k: np.concatenate(v) for k, v in stats.items()}
    dep = {k: np.concatenate(v) for k, v in dep.items()}

    calibration = np.zeros(requested, bool)
    for chain in range(n_chains):
        idx = np.flatnonzero(chain_of == chain)
        calibration[idx[::2]] = True
    evaluation = ~calibration

    labels = ["__GLOBAL__", *arrays["celltypes"]]
    rows = []

    def add(domain, obs_ct, rep_ct):
        obs_all = np.column_stack([obs_ct.sum(axis=1), obs_ct])
        rep_all = np.column_stack([rep_ct.sum(axis=1), rep_ct])
        for j, label in enumerate(labels):
            o, r = obs_all[:, j], rep_all[:, j]
            up = (1 + np.sum(r >= o)) / (len(r) + 1)
            lo = (1 + np.sum(r <= o)) / (len(r) + 1)
            rows.append({"domain": domain, "celltype": label,
                         "observed_median": float(np.median(o)), "replicated_median": float(np.median(r)),
                         "replicated_q025": float(np.quantile(r, 0.025)),
                         "replicated_q975": float(np.quantile(r, 0.975)),
                         "upper_p": float(up), "lower_p": float(lo),
                         "flag_upper": bool(up <= PPC_ALPHA)})

    def ct_counts(flags):
        return np.column_stack([flags[:, c].sum(axis=1) for c in columns_by_celltype])

    for domain, obs_key, rep_key in (("residual_dispersion", "obs_sd", "rep_sd"),
                                     ("studentized_residual_tails", "obs_tail", "rep_tail")):
        low, high = np.quantile(stats[rep_key][calibration], [0.025, 0.975], axis=0)
        obs_flags = (stats[obs_key][evaluation] < low) | (stats[obs_key][evaluation] > high)
        rep_flags = (stats[rep_key][evaluation] < low) | (stats[rep_key][evaluation] > high)
        add(domain, ct_counts(obs_flags), ct_counts(rep_flags))
        if domain == "studentized_residual_tails":
            obs_tail_rate = obs_flags.mean(axis=0)
    add("raw_outcome_dependence",
        np.broadcast_to(observed_raw_dep, (evaluation.sum(), n_c)), dep["rep_raw"][evaluation])
    add("conditional_residual_dependence", dep["obs_res"][evaluation], dep["rep_res"][evaluation])

    checks = pd.DataFrame(rows)
    feature_checks = pd.DataFrame({
        "global_feature": features,
        "global_celltype": np.asarray(arrays["celltypes"])[arrays["feature_to_celltype"]],
        "n_patients_fitted": observed_mask.sum(axis=1),
        "observed_tail_median": np.median(stats["obs_tail"], axis=0),
        "replicated_tail_q975": np.quantile(stats["rep_tail"], 0.975, axis=0),
        "tail_outside_rate": obs_tail_rate,
    })
    glob_tail = checks[(checks.domain == "studentized_residual_tails") & (checks.celltype == "__GLOBAL__")].iloc[0]
    summary = {
        "ppc_version": "draw-matched-incomplete-grid-v1", "family": FAMILY, "profile": PROFILE,
        "joint_draws": int(requested), "evaluation_draws": int(evaluation.sum()),
        "trace_sha256": sha256_file(TRACE_PATH),
        "formal": PROFILE != "smoke" and SAMPLER_OVERRIDE is None,
        "tail_upper_p": float(glob_tail.upper_p),
        "tail_adequacy_pass": bool(glob_tail.upper_p > PPC_ALPHA),
        "global_domains_flagged": checks.loc[(checks.celltype == "__GLOBAL__") & checks.flag_upper, "domain"].tolist(),
        "dispersion_note": ("residual_dispersion uses feature-specific sigma fitted to the same data, "
                            "so it is weak by construction; read lower_p as well as upper_p"),
        "dependence_note": ("correlations centre each feature on its fitted patients and zero-fill removed "
                            "cells; the same rule is applied to observed and replicated data"),
    }
    write_csv(checks, OUTPUT_DIR / "ppc_domain_checks.csv")
    write_csv(feature_checks, OUTPUT_DIR / "ppc_feature_checks.csv")
    write_json(summary, OUTPUT_DIR / "ppc_summary.json")
    print(checks[checks.celltype == "__GLOBAL__"].to_string(index=False))
    print(json.dumps(summary, indent=2))

# 8. Compare with the final fit and apply the prespecified decision rule
def local_null_and_bfdr(samples, ids, rope):
    local_null = np.mean(np.abs(samples) <= rope, axis=1)
    order = np.lexsort((ids, local_null))
    cumulative = np.cumsum(local_null[order]) / np.arange(1, len(ids) + 1)
    accepted = np.flatnonzero(cumulative <= Q_LEVEL)
    k = int(accepted[-1] + 1) if len(accepted) else 0
    selected = np.zeros(len(ids), bool)
    selected[order[:k]] = True
    return local_null, selected

if STAGE == "compare":
    pub_diag = RUN_ROOT / "publication" / "run_diagnostics.json"
    pub_ppc = RUN_ROOT / "publication" / "ppc_summary.json"
    for path in (pub_diag, pub_ppc):
        if not path.exists():
            raise FileNotFoundError(f"compare needs {path}")
    if not json.loads(pub_diag.read_text())["gate_pass"] and SAMPLER_OVERRIDE is None:
        raise RuntimeError("The publication sensitivity fit did not pass its convergence gate")

    beta = idata.posterior["feature_bmi_slope"].transpose("feature", "chain", "draw")
    ids = np.asarray(beta.coords["feature"], str)
    samples = np.asarray(beta).reshape(len(ids), -1)
    hdi = az.hdi(idata.posterior["feature_bmi_slope"], hdi_prob=0.95)["feature_bmi_slope"]
    hdi = hdi.sel(feature=ids).values
    local_null, selected = local_null_and_bfdr(samples, ids, PRIMARY_ROPE)
    sens = pd.DataFrame({
        "global_feature": ids,
        "sens_mean": samples.mean(axis=1), "sens_sd": samples.std(axis=1, ddof=1),
        "sens_hdi_low": hdi[:, 0], "sens_hdi_high": hdi[:, 1],
        "sens_probability_positive": (samples > 0).mean(axis=1),
        "sens_local_null": local_null, "sens_bfdr_selected": selected,
    })
    sens["sens_hdi_excludes_zero"] = (sens.sens_hdi_low > 0) | (sens.sens_hdi_high < 0)
    for rope in ROPE_GRID:
        sens[f"sens_selected_rope_{rope:.2f}"] = local_null_and_bfdr(samples, ids, rope)[1]

    table_path = ORIGINAL_RUN / "publication_report_v2" / "tables" / SPEC["original_table"]
    original = pd.read_csv(table_path)[[
        "global_feature", "CellType", "Signature", "posterior_mean", "hdi_95_low", "hdi_95_high",
        "local_null_probability_rope_0p10", "bfdr_selected", "hdi_95_excludes_zero"]]
    original = original.rename(columns={
        "posterior_mean": "orig_mean", "hdi_95_low": "orig_hdi_low", "hdi_95_high": "orig_hdi_high",
        "local_null_probability_rope_0p10": "orig_local_null", "bfdr_selected": "orig_bfdr_selected",
        "hdi_95_excludes_zero": "orig_hdi_excludes_zero"})
    merged = original.merge(sens, on="global_feature", how="inner", validate="one_to_one")
    if len(merged) != SPEC["expected_features"]:
        raise ValueError("Feature sets do not align")

    touched = set(full_frame.loc[full_frame["low_read"], "global_feature"])
    merged["affected_by_removal"] = merged["global_feature"].isin(touched)
    merged["delta_mean"] = merged.sens_mean - merged.orig_mean
    orig_width = merged.orig_hdi_high - merged.orig_hdi_low
    merged["delta_in_orig_sd"] = merged.delta_mean / (orig_width / 3.92)
    merged["hdi_width_ratio"] = (merged.sens_hdi_high - merged.sens_hdi_low) / orig_width
    merged["sign_flip"] = np.sign(merged.sens_mean) != np.sign(merged.orig_mean)
    merged["hdi_status_changed"] = merged.sens_hdi_excludes_zero != merged.orig_hdi_excludes_zero
    bmi_sd = float(cohort["bmi"].std(ddof=1))
    merged["orig_mean_per_5kg_m2"] = merged.orig_mean * 5 / bmi_sd
    merged["sens_mean_per_5kg_m2"] = merged.sens_mean * 5 / bmi_sd

    celltype = merged.groupby("CellType").agg(
        n_features=("global_feature", "size"),
        orig_mean_slope=("orig_mean", "mean"), sens_mean_slope=("sens_mean", "mean"),
        orig_hdi_excl=("orig_hdi_excludes_zero", "sum"), sens_hdi_excl=("sens_hdi_excludes_zero", "sum"),
        median_width_ratio=("hdi_width_ratio", "median")).reset_index()
    celltype["mean_slope_shift"] = celltype.sens_mean_slope - celltype.orig_mean_slope
    removed = dropped_cells.groupby("CellType").size().rename("cells_removed")
    celltype = celltype.merge(removed, on="CellType", how="left").fillna({"cells_removed": 0})
    nu_sens = idata.posterior["nu"].mean(("chain", "draw")).to_pandas()
    if SPEC["nu_scope"] == "celltype":
        celltype["sens_nu_mean"] = celltype["CellType"].map(
            {k.split("||", 1)[1]: v for k, v in nu_sens.items()})

    either_hdi = merged.orig_hdi_excludes_zero | merged.sens_hdi_excludes_zero
    criteria = {
        "bfdr_selection_changes": int((merged.orig_bfdr_selected != merged.sens_bfdr_selected).sum()),
        "sign_flips_among_hdi_features": int((merged.sign_flip & either_hdi).sum()),
        "hdi_status_change_fraction": float(merged.hdi_status_changed.sum() / max(1, either_hdi.sum())),
        "hdi_status_change_count": int(merged.hdi_status_changed.sum()),
        "max_celltype_mean_slope_shift": float(celltype.mean_slope_shift.abs().max()),
    }
    material = {
        "bfdr": criteria["bfdr_selection_changes"] > 0,
        "sign_flip": criteria["sign_flips_among_hdi_features"] > 0,
        "hdi_status": (criteria["hdi_status_change_fraction"] > DECISION_RULE["hdi_status_change_fraction_max"]
                       and criteria["hdi_status_change_count"] >= DECISION_RULE["hdi_status_change_count_min"]),
        "celltype_shift": criteria["max_celltype_mean_slope_shift"] > DECISION_RULE["celltype_mean_slope_shift_max"],
    }
    decision = {
        "family": FAMILY, "min_reads": MIN_READS, "criteria": criteria, "triggered": material,
        "decision": ("MATERIAL_CHANGE_READ_DEPTH_MODEL_WARRANTED" if any(material.values())
                     else "CONCLUSIONS_STABLE_REPORT_AS_SENSITIVITY"),
        "n_hdi_excluding_zero": {"original": int(merged.orig_hdi_excludes_zero.sum()),
                                 "sensitivity": int(merged.sens_hdi_excludes_zero.sum())},
        "n_bfdr_selected": {"original": int(merged.orig_bfdr_selected.sum()),
                            "sensitivity": int(merged.sens_bfdr_selected.sum())},
        "sensitivity_selected_across_rope_grid": {
            f"{r:.2f}": int(merged[f"sens_selected_rope_{r:.2f}"].sum()) for r in ROPE_GRID},
        "nu_sensitivity": nu_sens.to_dict() if SPEC["nu_scope"] == "celltype" else float(nu_sens),
        "ppc_summary": json.loads(pub_ppc.read_text()),
        "original_table_sha256": sha256_file(table_path),
        "sensitivity_trace_sha256": sha256_file(RUN_ROOT / "publication" / "posterior_trace.nc"),
    }
    compare_dir = RUN_ROOT / "comparison"
    compare_dir.mkdir(exist_ok=True)
    write_csv(merged, compare_dir / "feature_comparison.csv")
    write_csv(celltype, compare_dir / "celltype_comparison.csv")
    write_json(decision, compare_dir / "decision.json")

    print(json.dumps({k: decision[k] for k in ("decision", "criteria", "triggered",
                                                "n_hdi_excluding_zero", "n_bfdr_selected")}, indent=2))
    print(celltype.round(3).to_string(index=False))
    cols = ["CellType", "Signature", "orig_mean", "sens_mean", "delta_in_orig_sd",
            "hdi_width_ratio", "orig_hdi_excludes_zero", "sens_hdi_excludes_zero"]
    print(merged.reindex(merged.delta_mean.abs().sort_values(ascending=False).index)[cols]
          .head(15).round(3).to_string(index=False))


In [ ]:
# 4. Run everything sequentially (resumable)
import gc, json, time, traceback, zipfile
from pathlib import Path

FAMILIES = ["immune_fine"]   
SKIP_STANDARD = True        # go smoke -> publication with a recorded waiver
DOWNLOAD_AT_END = True
ENGINE_PATH = globals().get("ENGINE_PATH", "/content/lowread_engine.py")
TEST_CONFIG = globals().get("TEST_CONFIG", {})   # offline testing only; leave empty on Colab

ENGINE_SRC = Path(ENGINE_PATH).read_text()
BFMI_MIN, MAXDEPTH_MAX = 0.30, 0.01
RESULT_FILES = [
    "preflight_manifest.json", "dropped_low_read_cells.csv",
    *[f"{p}/{f}" for p in ("smoke", "standard", "publication")
      for f in ("run_diagnostics.json", "diagnostic_families.csv", "promotion_waiver.json",
                "ppc_summary.json", "ppc_domain_checks.csv", "ppc_feature_checks.csv")],
    "comparison/decision.json", "comparison/feature_comparison.csv",
    "comparison/celltype_comparison.csv",
]

def run_step(family, profile, stage, waiver=None):
    """Execute the engine once in a fresh namespace; return that namespace."""
    config = {"family": family, "profile": profile, "stage": stage, "waiver": waiver, **TEST_CONFIG}
    ns = {"CONFIG": config, "__name__": "lowread_engine"}
    started = time.time()
    print(f"\n{'=' * 70}\n{family} | {profile} | {stage}\n{'=' * 70}", flush=True)
    exec(compile(ENGINE_SRC, "lowread_engine.py", "exec"), ns)
    print(f"[{family} {profile} {stage}] finished in {(time.time() - started) / 60:.1f} min", flush=True)
    return ns

def read_json(path):
    return json.loads(path.read_text()) if path.exists() else None

def paths_for(family):
    ns = run_step(family, "smoke", "preflight")      # validates inputs and resolves paths
    root = Path(ns["RUN_ROOT"])
    info = {"root": root, "run_name": ns["RUN_NAME"], "sens_root": Path(ns["SENS_ROOT"])}
    del ns
    gc.collect()
    return info

def gate_ok(record):
    if record is None:
        return False
    return bool(record["gate_pass"] or (TEST_CONFIG.get("sampler_override") is not None))

def geometry_clean(record):
    return bool(record and record["divergences"] == 0 and record["min_bfmi"] >= BFMI_MIN
                and record["max_maxdepth_fraction"] <= MAXDEPTH_MAX
                and record["nonfinite_diagnostics"] == 0 and record["max_rhat"] < 1.10)

def zip_results(info):
    zip_dir = info["sens_root"] / "_result_zips"
    zip_dir.mkdir(parents=True, exist_ok=True)
    out = zip_dir / f"{info['run_name']}_results.zip"
    missing = []
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for rel in RESULT_FILES:
            src = info["root"] / rel
            if src.exists():
                z.write(src, f"{info['run_name']}/{rel}")
            elif not (rel.startswith("standard/") or "promotion_waiver" in rel
                      or (rel.startswith("smoke/") and "ppc" in rel)):
                missing.append(rel)
        trace = info["root"] / "publication" / "posterior_trace.nc"
        if trace.exists():
            import arviz as az
            summary = az.summary(az.from_netcdf(trace),
                                 var_names=["celltype_bmi_slope", "feature_sigma_bmi", "nu",
                                            "patient_global_sigma", "log_sigma_celltype_location"],
                                 kind="all", round_to=6)
            z.writestr(f"{info['run_name']}/publication/key_parameter_summary.csv", summary.to_csv())
        z.writestr(f"{info['run_name']}/driver_log.json", json.dumps(LOG.get(info["run_name"], []), indent=2))
    return out, missing

def run_family(family, info):
    root, log = info["root"], LOG.setdefault(info["run_name"], [])
    note = lambda msg: (log.append({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "event": msg}), print("»", msg))

    # 1. Smoke fit (skipped if already done)
    smoke = read_json(root / "smoke" / "run_diagnostics.json")
    if smoke is None:
        if (root / "smoke" / "posterior_trace.nc").exists():
            run_step(family, "smoke", "diagnostics")
        else:
            run_step(family, "smoke", "fit")
        gc.collect()
        smoke = read_json(root / "smoke" / "run_diagnostics.json")
    else:
        note("smoke fit already done; reusing it")
    note(f"smoke gate_pass={smoke['gate_pass']} max_rhat={smoke['max_rhat']:.4f} "
         f"min_ess_bulk={smoke['min_ess_bulk']:.0f} min_ess_tail={smoke['min_ess_tail']:.0f} "
         f"divergences={smoke['divergences']}")
    if not geometry_clean(smoke) and not TEST_CONFIG:
        note("STOP: smoke fit shows a sampler-geometry problem; no publication fit for this family")
        return "stopped_smoke_geometry"

    # 2. Standard (optional) and the promotion waiver
    waiver = None
    if SKIP_STANDARD:
        waiver = (f"Standard stage skipped by protocol. Smoke: gate_pass={smoke['gate_pass']}, "
                  f"max R-hat {smoke['max_rhat']:.3f}, min ESS bulk/tail "
                  f"{smoke['min_ess_bulk']:.0f}/{smoke['min_ess_tail']:.0f}, 0 divergences, "
                  f"min BFMI {smoke['min_bfmi']:.2f}. Publication gate is binding.")
    else:
        standard = read_json(root / "standard" / "run_diagnostics.json")
        if standard is None:
            if not smoke["gate_pass"]:
                waiver = (f"Smoke failed on ESS only (min {min(smoke['min_ess_bulk'], smoke['min_ess_tail']):.0f}) "
                          "with clean geometry.")
            run_step(family, "standard", "fit", waiver)
            gc.collect()
            standard = read_json(root / "standard" / "run_diagnostics.json")
        waiver = None if standard["gate_pass"] else (
            f"Standard failed on ESS/R-hat (max R-hat {standard['max_rhat']:.3f}, min ESS "
            f"{min(standard['min_ess_bulk'], standard['min_ess_tail']):.0f}) with "
            f"{standard['divergences']} divergences.")

    # 3. Publication fit (skipped if trace exists)
    pub_dir = root / "publication"
    if (pub_dir / "posterior_trace.nc").exists():
        note("publication trace exists; not refitting")
        if read_json(pub_dir / "run_diagnostics.json") is None:
            run_step(family, "publication", "diagnostics", waiver)
    else:
        run_step(family, "publication", "fit", waiver)
    gc.collect()
    pub = read_json(pub_dir / "run_diagnostics.json")
    note(f"publication gate_pass={pub['gate_pass']} max_rhat={pub['max_rhat']:.4f} "
         f"min_ess_bulk={pub['min_ess_bulk']:.0f} min_ess_tail={pub['min_ess_tail']:.0f} "
         f"divergences={pub['divergences']}")
    if not gate_ok(pub):
        note("STOP: publication gate failed; PPC and comparison not run for this family")
        return "stopped_publication_gate"

    # 4. PPC
    if read_json(pub_dir / "ppc_summary.json") is None:
        run_step(family, "publication", "ppc", waiver)
        gc.collect()
    else:
        note("publication PPC already done")
    ppc = read_json(pub_dir / "ppc_summary.json")
    note(f"PPC tail_upper_p={ppc['tail_upper_p']:.4f} flagged={ppc['global_domains_flagged']}")

    # 5. Compare
    if read_json(root / "comparison" / "decision.json") is None:
        run_step(family, "publication", "compare", waiver)
        gc.collect()
    else:
        note("comparison already done")
    decision = read_json(root / "comparison" / "decision.json")
    note(f"DECISION: {decision['decision']} | triggered={decision['triggered']}")
    return decision["decision"]

LOG, OUTCOMES, ZIPS = {}, {}, []
for family in FAMILIES:
    info = None
    try:
        info = paths_for(family)
        OUTCOMES[family] = run_family(family, info)
    except Exception as error:
        OUTCOMES[family] = f"ERROR: {type(error).__name__}: {error}"
        traceback.print_exc()
        print(f"{family} failed; continuing with the next family")
    finally:
        if info is not None:
            try:
                path, missing = zip_results(info)
                ZIPS.append(path)
                print(f"Zip saved: {path} | missing: {missing or 'none'}")
            except Exception as error:
                print(f"Could not zip {family}: {error}")
        gc.collect()

print("\n" + "=" * 70)
for family, outcome in OUTCOMES.items():
    print(f"{family:15s} {outcome}")

if DOWNLOAD_AT_END and ZIPS:
    try:
        from google.colab import files
        bundle = Path("/content/lowread_sensitivity_all_results.zip")
        with zipfile.ZipFile(bundle, "w") as z:
            for path in ZIPS:
                z.write(path, path.name)
        files.download(str(bundle))
    except ImportError:
        print("Not in Colab; zips are in", ZIPS[0].parent)
